# 04.4 · q-transfer — cross-task multi-predicate transplant

Clean copy of the last section of `04_q_transfer.ipynb`. SRC and DST list **different** task sets that share some
tasks. SRC is about to write its next task (the **target**), which DST does not have; DST is at its own frontier.
Both stop at the same pivot (the trailing indent). We cache the pre-RoPE **query** of the patch heads at the SRC pivot,
inject it at the DST pivot (prefill), and re-clamp it at the first `N_STEER` decode steps. Does DST now pursue SRC's
target instead of its own frontier?

Tasks are letters: `A, B, C, …` name predicates (knob `TASKS`). The result variables in the code are `a, b, c, …` by
position, which is unrelated to the task letters.

Per run (source / destination / patched) the notebook shows the greedy generation, the GoalScope read-out at the pivot,
and the pivot's **gaze** (value-weighted attention) over the listed tasks.

**Optional kv-transfer (`USE_KV`, the 04.3 setup):** a third prompt, KV, lists `KV_ORDER` with DST's own tasks written.
Its task-list key+value on `KV_HEADS` are transplanted onto DST's task-list span (GQA-corrected head-wise routing,
`pad` alignment), so DST "sees" KV's tasks; `KV_ALPHA` boosts the patched heads' OV write at the current token on
every patched forward. Every patched run below (q-patch, read-outs, closed loop) then runs on the kv-patched DST, and
the gaze is read per **KV item** mapped onto DST's span. kv-only expects KV's next task; kv+q under H1 expects the
first KV task SRC has not done; SRC's own target identity expects SRC's target.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys

CUDA = "7"                                   # free GPU(s); Coder-32B fits on one 80 GB GPU
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("../..")                     # repo root, so `src` is importable
import json, torch, numpy as np
import src.diff.functional as FUNC
from src.diff.functional import *            # helpers into scope
import src.diff.data as DATA
from src.attention import get_attention_matrices
from rich.panel import Panel
from rich.table import Table
from rich.text import Text
from rich.markup import escape
from src.utils.experiment_utils import set_seed

FUNC.setup("Qwen/Qwen2.5-Coder-32B-Instruct")   # loads the model + the 122 goal heads (lam20 · thr 0.9 · 25-56)
mt = FUNC.mt


def pivot_of(prompt, neg_shift=-1):
    return prepare_input(prompts=prompt, tokenizer=mt)["input_ids"].shape[1] + neg_shift


def show(prompt, title, emph_span=None, piv=-1):
    kw = dict(patch_pos=[piv])
    if emph_span is not None:
        kw["emphasize_tokens"] = range(*emph_span)
    rich_print(Panel(Text.from_markup(exact_prompt(prompt, **kw)), title=f"[bold]{title}[/bold]",
                     title_align="left", border_style="grey42", padding=(1, 1)))

2026-10-06 15:00:18 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


## Knobs

In [20]:
# ── knobs ────────────────────────────────────────────────────────────────────
N_GOAL_HEADS = 250        # 122: lam20 · thr 0.9 · layers 25-56 (paper set) | 250: lam20 top-250 (best kv transfer)
DATATYPE     = "list[str]"
NAMING       = "letters"  # result variable names in the code: "letters" (a, b, c) | "natural" (descriptive)

TASKS = {                 # task letter -> predicate id in DATA.QTASKS[DATATYPE]
    "B": "count_long",
    "A": "count_capitalized",
    "C": "distinct_count",
    "D": "count_starts_vowel",
    "X": "total_length",       # x: DST's own frontier (word-length family: distinct from b / c / d, as in 04.3)
    "Y": "has_palindrome",     # y: DST's 2nd pending task
}
# Design (04.3 notation, ' = written):  p_src a'cb' -> c   p_dst a'xy -> x   p_kv a'bd -> b (its prefill is irrelevant)
#   kv-only -> b  ·  kv+q: set progress -> d  ({a,b,d} minus DST's a and SRC's {a,b})  ·  count -> y (slot |SRC_DONE| of DST)
SRC_ORDER, SRC_DONE = "ACB", "AB"   # SRC: listed tasks (in order), written tasks -> target = first listed not written
DST_ORDER, DST_DONE = "AXY", "A"    # DST: listed tasks (in order), written tasks -> frontier = first listed not written

PIVOT    = "var_eq"       # "indent": prefill ends at the trailing indent · "var_eq": ends at the next var's " =" (`c =`)
PREFILL_ANCHORS = ("indent", "var", "eq")   # prefill: SRC query at each anchor -> the SAME anchor in DST ("var"/"eq" need PIVOT="var_eq")
DECODE_Q = "eq"          # query re-clamped at decode steps: "eq" (SRC " =" query) | "mean" (avg of SRC indent/var/eq queries)

# PIVOT = "indent"
# PREFILL_ANCHORS = ("indent",)
# DECODE_Q = "indent"

N_STEER  = 32             # forward passes that carry the SRC query: 1 = DST pivot only (prefill); k = pivot + first k-1 decode steps
MAX_NEW  = 96             # generation budget per run
USE_KV   = True          # True: also transplant KV's task-list k+v onto DST's task list (04.3 setup)
KV_ORDER = "ABD"          # KV's task list; its written tasks = DST_DONE, so only the task list differs from DST
KV_HEADS = None           # heads whose k+v are transplanted (None = GOAL_HEADS)
KV_ALPHA = 2.0            # OV-write boost of KV_HEADS ∪ PATCH_HEADS at the current token on patched runs (1.0 = off)
KV_ALIGN = "pad"          # kv span alignment: "pad" | "right"
RO_KW    = dict(strategy="additive_norm", alpha=0.1, layers=(35, 46))   # GoalScope read-out config (paper default)

# ── derived ──────────────────────────────────────────────────────────────────
if N_GOAL_HEADS == 122:
    GOAL_HEADS = list(FUNC.GOAL_HEADS)
elif N_GOAL_HEADS == 250:
    GOAL_HEADS = [tuple(h) for h in json.load(open(
        "../../head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json"))]
else:
    raise ValueError("N_GOAL_HEADS must be 122 or 250")

READOUT_HEADS = GOAL_HEADS

#! overwrite goal heads
# GOAL_HEADS = [(l, h) for l in range(35, 55) for h in range(mt.n_heads)]

PATCH_HEADS = GAZE_HEADS = GOAL_HEADS 
KV_HEADS = KV_HEADS or GOAL_HEADS

bank = DATA.QTASKS[DATATYPE]
by = {p.id: p for p in bank.predicates}
assert all(TASKS[L] in by for L in SRC_ORDER + DST_ORDER), "unknown predicate id in TASKS"
assert set(SRC_DONE) <= set(SRC_ORDER) and set(DST_DONE) <= set(DST_ORDER)
TARGET   = next(L for L in SRC_ORDER if L not in SRC_DONE)    # SRC's next task
FRONTIER = next(L for L in DST_ORDER if L not in DST_DONE)    # DST's own next task
TN = lambda L: f"{L} = {TASKS[L]!r}" if L else "None"   # label: letter + task name
if USE_KV:
    assert set(DST_DONE) <= set(KV_ORDER) and all(TASKS[L] in by for L in KV_ORDER), "KV_ORDER must contain DST_DONE"
    KV_NEXT = next(L for L in KV_ORDER if L not in DST_DONE)                         # kv-only expectation
    # kv+q under set progress: first patched-list (KV) task done in neither DST nor SRC
    H1_NEXT = next((L for L in KV_ORDER if L not in set(DST_DONE) | set(SRC_DONE)), None)
    # kv+q under count: DST slot |SRC_DONE| — read on DST's own listing (on the kv-patched list it would be
    # KV_ORDER[|SRC_DONE|], which here coincides with set progress)
    COUNT_NEXT = DST_ORDER[len(SRC_DONE)] if len(SRC_DONE) < len(DST_ORDER) else None
    print(f"KV {{{','.join(KV_ORDER)}}} · kv-only expects {TN(KV_NEXT)} · kv+q: set progress -> {TN(H1_NEXT)}, "
          f"count -> {TN(COUNT_NEXT)} (SRC identity -> {TN(TARGET)})")
print(f"{len(GOAL_HEADS)} goal heads · SRC target {TN(TARGET)} · DST frontier {TN(FRONTIER)} · "
      f"target in DST: {TARGET in DST_ORDER} · N_STEER={N_STEER}")
for L in sorted(set(SRC_ORDER + DST_ORDER + (KV_ORDER if USE_KV else ""))):
    print(f"  {L} = {TASKS[L]:<22} {by[TASKS[L]].description}")

KV {A,B,D} · kv-only expects B = 'count_long' · kv+q: set progress -> D = 'count_starts_vowel', count -> Y = 'has_palindrome' (SRC identity -> C = 'distinct_count')
250 goal heads · SRC target C = 'distinct_count' · DST frontier X = 'total_length' · target in DST: False · N_STEER=32
  A = count_capitalized      the number of capitalized words
  B = count_long             the number of words longer than five characters
  C = distinct_count         the number of distinct words
  D = count_starts_vowel     the number of words that start with a vowel
  X = total_length           the total number of characters across all words
  Y = has_palindrome         whether any word is a palindrome


## SRC and DST prompts

In [21]:
# ── SRC / DST prompts ────────────────────────────────────────────────────────
class SeqQTI(DATA.QTransferInstance):
    """`letters` naming is SEQUENTIAL (DATA._seq_letter): prefilled lines are a, b, c, ... in the order they are
    written and the frontier stub takes the next letter — never the task's list position. (The base
    QTransferInstance is still positional, which leaks list position whenever the prefill is not a list prefix.)"""
    def var_of(self, pred_id):
        if self.naming != "letters":
            return super().var_of(pred_id)
        return DATA._seq_letter(self.task.predicates, self.src_progress, pred_id)


def make_inst(order, done, target):
    task = DATA.dataclasses.replace(bank, id="xtask", predicates=tuple(by[TASKS[L]] for L in order))
    ids = [TASKS[L] for L in order if L in done]      # written in list order (matches _seq_letter)
    return SeqQTI(task=task, var_name=bank.var_names[0], presentation="listed",
                                  src_lang="python", dst_lang="python", src_progress=ids, dst_progress=ids,
                                  expected_next=TASKS[target], naming=NAMING, meta={"datatype": DATATYPE})


src_inst = make_inst(SRC_ORDER, SRC_DONE, TARGET)
dst_inst = make_inst(DST_ORDER, DST_DONE, FRONTIER)
def build_prompt(inst, parts, next_task):
    q, pf = parts
    if PIVOT == "var_eq":            # stub the next task's (sequential) variable up to " =" -> pivot = " ="
        pf = pf + f"{inst.var_of(TASKS[next_task])} ="
    elif PIVOT != "indent":
        raise ValueError(f"PIVOT must be 'indent' or 'var_eq', got {PIVOT!r}")
    return get_prompt(q, DATA._finalize_prefill(pf))


src_prompt = build_prompt(src_inst, src_inst.src_parts(), TARGET)
dst_prompt = build_prompt(dst_inst, dst_inst.dst_parts(), FRONTIER)
piv_src, piv_dst = pivot_of(src_prompt), pivot_of(dst_prompt)
src_tok = prepare_input(prompts=src_prompt, tokenizer=mt)
dst_tok = prepare_input(prompts=dst_prompt, tokenizer=mt)

tasks_str = lambda order: "{" + ",".join(order) + "}"
print(f"pivot token: SRC {mt.tokenizer.decode(src_tok['input_ids'][0][piv_src:piv_src+1])!r} · "
      f"DST {mt.tokenizer.decode(dst_tok['input_ids'][0][piv_dst:piv_dst+1])!r}")
show(src_prompt, f"SRC · {tasks_str(SRC_ORDER)} · {','.join(SRC_DONE)} done · target {TN(TARGET)} · pivot@{piv_src}")
if not USE_KV:                                       # with kv, DST is shown aligned (with spans) below
    show(dst_prompt, f"DST · {tasks_str(DST_ORDER)} · {','.join(DST_DONE)} done · frontier {TN(FRONTIER)} · pivot@{piv_dst}")

# ── optional KV prompt (04.3): its task-list k+v -> DST's task list ───────────
dst_tok_clean, piv_dst_clean = dst_tok, piv_dst      # clean DST, never padded
task_text = lambda order: "\n".join(f"({i + 1}) {by[TASKS[L]].description}" for i, L in enumerate(order))


def item_positions(prompt, order):
    return {L: list(range(*token_span(prompt, by[TASKS[L]].description))) for L in order}


PATCHED_ITEMS = item_positions(dst_prompt, DST_ORDER)   # {letter: DST positions} the patched runs' gaze is read over
_FROM_DST = {}                                           # DST position -> p_kv position, inside the kv window
PATCHED_LABEL = "DST tasks"


def show_spans(tok, title, task_span=None, kv_span=None, pad=(), piv=-1):
    """Render a tokenized prompt exactly as the model sees it (from 04.3): task_span -> bright_yellow,
    kv_span -> black on dark_orange (the exact tokens whose k+v move), pad -> the spliced "\\n" alignment tokens
    (black on grey62, shown as ⏎pad), pivot -> salmon. Priority pivot > pad > kv > task."""
    ids = tok["input_ids"][0].tolist()
    n = len(ids); pivpos = n + piv if piv < 0 else piv
    task = set(range(*task_span)) if task_span else set()
    kv = set(range(*kv_span)) if kv_span else set()
    pad = set(pad)
    # sentinel markers, escape the joined string once, then swap in rich tags (per-token escaping drops brackets)
    PO, PC, DO, DC, KO, KC, TO, TC = "\x00<", "\x00>", "\x03<", "\x03>", "\x01<", "\x01>", "\x02<", "\x02>"
    parts = []
    for i in range(n):
        s = mt.tokenizer.decode([ids[i]]).replace("\n", "\\n\n")
        if i == pivpos:
            s = PO + s + PC
        elif i in pad:
            s = DO + "⏎pad" + DC + "\n"
        elif i in kv:
            s = KO + s + KC
        elif i in task:
            s = TO + s + TC
        parts.append(s)
    txt = escape("".join(parts))
    for a, b, tag in ((PO, PC, "bold black on salmon1"), (DO, DC, "black on grey62"),
                      (KO, KC, "bold black on dark_orange"), (TO, TC, "bright_yellow")):
        txt = txt.replace(a, f"[{tag}]").replace(b, f"[/{tag}]")
    rich_print(Panel(Text.from_markup(txt), title=f"[bold]{escape(title)}[/bold]",
                     title_align="left", border_style="grey42", padding=(1, 1)))


def patched_tok_text(j):
    # token at DST position j as the patched runs' goal heads see it: inside the kv window that is p_kv's token
    # (its k+v were transplanted there), elsewhere DST's own
    if j in _FROM_DST:
        return mt.tokenizer.decode([int(kv_tok["input_ids"][0][_FROM_DST[j]])])
    return mt.tokenizer.decode([int(dst_tok["input_ids"][0][j])])


if USE_KV:
    kv_inst = make_inst(KV_ORDER, DST_DONE, KV_NEXT)
    kv_prompt = build_prompt(kv_inst, kv_inst.dst_parts(), KV_NEXT)
    kv_task, dst_task = task_text(KV_ORDER), task_text(DST_ORDER)
    KV0, DST0 = token_span(kv_prompt, kv_task), token_span(dst_prompt, dst_task)
    kv_tok, dst_tok, KV_POS, DST_POS, KV_SPAN, DST_SPAN = align_kv_spans(
        mt, kv_prompt, dst_prompt, KV0, DST0, mode=KV_ALIGN, src_text=kv_task, dst_text=dst_task)
    piv_dst = dst_tok["input_ids"].shape[1] - 1          # patched runs use the (possibly padded) aligned DST
    _kvh = heads_per_layer(KV_HEADS, "k", layers=set(FUNC._layers_of(KV_HEADS)))
    KV_K = cache_src_proj(kv_tok, KV_POS, sorted(_kvh), "k")
    KV_V = cache_src_proj(kv_tok, KV_POS, sorted(_kvh), "v")
    # KV items -> DST positions through the aligned window (pad splices newlines right before the shorter span)
    _kv_pad = (KV_SPAN[1] - KV_SPAN[0]) - (KV0[1] - KV0[0])
    _to_dst = dict(zip(KV_POS, DST_POS))
    _FROM_DST = dict(zip(DST_POS, KV_POS))
    PATCHED_ITEMS = {L: [_to_dst[p + _kv_pad] for p in ps if p + _kv_pad in _to_dst]
                     for L, ps in item_positions(kv_prompt, KV_ORDER).items()}
    PATCHED_LABEL = "p_kv tasks"
    # alignment pads: "pad" splices newline ids right BEFORE the shorter span -> the first n positions of that span
    _dst_pad = (DST_SPAN[1] - DST_SPAN[0]) - (DST0[1] - DST0[0])
    KV_PADS = list(range(KV_SPAN[0], KV_SPAN[0] + _kv_pad)) if KV_ALIGN == "pad" else []
    DST_PADS = list(range(DST_SPAN[0], DST_SPAN[0] + _dst_pad)) if KV_ALIGN == "pad" else []
    print(f"align={KV_ALIGN}: {len(KV_POS)} transplanted tokens · pads: p_kv {len(KV_PADS)} · p_dst {len(DST_PADS)}"
          f" · same positions: {KV_POS == DST_POS}")
    show_spans(dst_tok, f"p_dst · {tasks_str(DST_ORDER)} · {','.join(DST_DONE)} done · frontier {TN(FRONTIER)} · "
                        f"k/v PATCHED-TO (aligned) · pivot@{piv_dst}",
               task_span=DST_SPAN, kv_span=(DST_POS[0], DST_POS[-1] + 1), pad=DST_PADS)
    show_spans(kv_tok, f"p_kv · {tasks_str(KV_ORDER)} · kv-only expects {TN(KV_NEXT)} · k/v CACHED-FROM (aligned)",
               task_span=KV_SPAN, kv_span=(KV_POS[0], KV_POS[-1] + 1), pad=KV_PADS)

pivot token: SRC ' =' · DST ' ='


╭─ SRC · {A,C,B} · A,B done · target C = 'distinct_count' · pivot@146 ────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of capitalized words\n                                                                           │
│ (2) the number of distinct words\n                                                                              │
│ (3) the number of words longer than five characters\n                                                           │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(1 for w in words if w[:1].isupper())\n                                                              │
│     b = sum(1 for w in words if len(w) > 5)\n                                                                   │
│     c =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

align=pad: 33 transplanted tokens · pads: p_kv 0 · p_dst 3 · same positions: True


╭─ p_dst · {A,X,Y} · A done · frontier X = 'total_length' · k/v PATCHED-TO (aligned) · pivot@132 ─────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ ⏎pad                                                                                                            │
│ ⏎pad                                                                                                            │
│ ⏎pad                                                                                                            │
│ (1) the number of capitalized words\n                                                                           │
│ (2) the total number of characters across all words\n                                                           │
│ (3) whether any word is a palindrome\n                                                                          │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(1 for w in words if w[:1].isupper())\n                                                              │
│     b =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ p_kv · {A,B,D} · kv-only expects B = 'count_long' · k/v CACHED-FROM (aligned) ─────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of capitalized words\n                                                                           │
│ (2) the number of words longer than five characters\n                                                           │
│ (3) the number of words that start with a vowel\n                                                               │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(1 for w in words if w[:1].isupper())\n                                                              │
│     b =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Runs — source / destination / patched

In [22]:
# ── source / destination / patched runs: generation + GoalScope read-out at the pivot ──
Q_LAYERS = sorted({l for l, _ in PATCH_HEADS})
_qh = heads_per_layer(PATCH_HEADS, "q")

# anchors = the last tokens of the prefill: "    c =" -> ['   ', ' c', ' ='] (indent, var, eq); indent pivot -> ['    ']
ANCHOR_OFFSET = {"indent": -3, "var": -2, "eq": -1} if PIVOT == "var_eq" else {"indent": -1}
assert set(PREFILL_ANCHORS) <= set(ANCHOR_OFFSET), f"PREFILL_ANCHORS {PREFILL_ANCHORS} need PIVOT='var_eq'"
anchor_pos = lambda tok: {a: tok["input_ids"].shape[1] + o for a, o in ANCHOR_OFFSET.items()}
SRC_ANCH, DST_ANCH = anchor_pos(src_tok), anchor_pos(dst_tok)
_tok_at = lambda tok, p: mt.tokenizer.decode([int(tok["input_ids"][0][p])])
for side, tok, anch in (("SRC", src_tok, SRC_ANCH), ("DST", dst_tok, DST_ANCH)):
    print(f"{side} anchors: " + "  ".join(f"{a}@{p}={_tok_at(tok, p)!r}" for a, p in anch.items()))
if PIVOT == "var_eq":
    assert all(_tok_at(t, a["indent"]).strip() == "" and _tok_at(t, a["eq"]).strip() == "="
               for t, a in ((src_tok, SRC_ANCH), (dst_tok, DST_ANCH))), "anchor offsets do not match the tokens"

# SRC queries (pre-RoPE) at every anchor: {layer: (n_anchors, width)}, rows in ANCHOR_OFFSET order
_anchors = list(ANCHOR_OFFSET)
q_src_all = cache_src_proj(src_prompt, [SRC_ANCH[a] for a in _anchors], Q_LAYERS, "q")
q_rows = lambda names: {l: c[[_anchors.index(a) for a in names]] for l, c in q_src_all.items()}

# prefill: row r of the cache goes to DST position r — indent->indent, var->var, eq->eq
PF = [dict(which="q", cache=q_rows(PREFILL_ANCHORS), heads=_qh,
           positions=[DST_ANCH[a] for a in PREFILL_ANCHORS])]
# decode: one query re-clamped at the current token for the first N_STEER-1 decode steps
if DECODE_Q == "eq" or PIVOT == "indent":
    q_dec = q_rows([_anchors[-1]])                                       # the " =" (or indent) query
elif DECODE_Q == "mean":
    q_dec = {l: c.mean(0, keepdim=True) for l, c in q_src_all.items()}  # mean of indent / var / eq
else:
    raise ValueError(f"DECODE_Q must be 'eq' or 'mean', got {DECODE_Q!r}")
DC = [dict(which="q", cache=q_dec, heads=_qh, positions=[-1])]
from contextlib import ExitStack


def patched_ctx(pf=None, boost=True):
    # stack: kv transplant (USE_KV) + query patch `pf` + the 04.3 OV boost (USE_KV only) — for ONE patched forward
    s = ExitStack()
    if USE_KV:
        s.enter_context(headwise_kv_patch(mt, KV_HEADS, KV_K, KV_V, DST_POS))
    if pf:
        s.enter_context(proj_patch_hooks(mt, pf))
    if boost and USE_KV and KV_ALPHA != 1.0:
        s.enter_context(boost_head_ov(mt, sorted(set(KV_HEADS) | set(PATCH_HEADS)), KV_ALPHA, positions=(-1,)))
    return s


def ro_iface(pf=None, boost=True):
    def _if(mt, inputs, patches=None, model_kwargs=None):
        with patched_ctx(pf, boost=boost):
            return mt._model(**dict(inputs), **dict(model_kwargs or {}))
    return _if


IFACE = ro_iface(PF)                       # kv (if USE_KV) + prefill q-patch (+ boost), for read-outs
EXPECT_PATCHED = f"set {TN(H1_NEXT)} / count {TN(COUNT_NEXT)}" if USE_KV else TN(TARGET)   # what the patched run is judged against
print(f"prefill patch at {list(PREFILL_ANCHORS)} · decode query = {DECODE_Q if PIVOT == 'var_eq' else 'indent'} "
      f"· N_STEER={N_STEER}")

GEN, RO = {}, {}
GEN["source"] = greedy(src_tok, max_new=MAX_NEW)
rich_print(f"[red1]source · clean SRC (target {TN(TARGET)}):[/red1]\n{escape(GEN['source'])}")
RO["source"] = goalscope_readout("   ↳ goalscope · source pivot", src_prompt, piv_src, heads=READOUT_HEADS, **RO_KW)

GEN["destination"] = greedy(dst_tok_clean, max_new=MAX_NEW)
rich_print(f"[sea_green3]destination · clean DST (frontier {TN(FRONTIER)}):[/sea_green3]\n{escape(GEN['destination'])}")
RO["destination"] = goalscope_readout("   ↳ goalscope · destination pivot", dst_prompt, piv_dst_clean,
                                      heads=READOUT_HEADS, **RO_KW)

if USE_KV:
    with patched_ctx():
        GEN["kv"] = greedy(dst_tok, max_new=MAX_NEW)
    rich_print(f"[orange3]kv-only · DST + KV task-list k+v (expect {TN(KV_NEXT)}):[/orange3]\n{escape(GEN['kv'])}")
    RO["kv"] = goalscope_readout("   ↳ goalscope · kv-only pivot", dst_tok, piv_dst, heads=READOUT_HEADS,
                                 iface=ro_iface(), **RO_KW)

with patched_ctx():                        # kv + boost; the query patch itself comes from greedy's specs
    GEN["patched"] = greedy(dst_tok, prefill_specs=PF, decode_specs=DC, n_steer=N_STEER, max_new=MAX_NEW)
rich_print(f"[salmon1]patched · DST{' + KV k+v' if USE_KV else ''} + SRC query (expect {EXPECT_PATCHED}; "
           f"N_STEER={N_STEER}):[/salmon1]\n"
           f"{escape(GEN['patched'])}")
RO["patched"] = goalscope_readout(f"   ↳ goalscope · patched pivot (expect {EXPECT_PATCHED})", dst_tok, piv_dst,
                                  heads=READOUT_HEADS,
                                  iface=IFACE, **RO_KW)

SRC anchors: indent@144='   '  var@145=' c'  eq@146=' ='
DST anchors: indent@130='   '  var@131=' b'  eq@132=' ='
prefill patch at ['indent', 'var', 'eq'] · decode query = eq · N_STEER=32


source · clean SRC (target C = 'distinct_count'):
 len(set(words))
    return a, c, b
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · source pivot  pos=146 ' =' => I'm counting unique elements in a list using a set.

destination · clean DST (frontier X = 'total_length'):
 sum(len(w) for w in words)
    c = any(w == w[::-1] for w in words)
    return a, b, c
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

... (3 outputs left out)

   ↳ goalscope · kv-only pivot  pos=132 ' =' => I'm checking if the length of the string is greater than 5.

patched · DST + KV k+v + SRC query (expect set D = 'count_starts_vowel' / count Y = 'has_palindrome'; N_STEER=32):
 sum(1 for w in words if len(w) > 0)
    c = sum(1 for w in words if w[0].isalpha() and w[0].lower() in 'aeiou')
    return a, b, c
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · patched pivot (expect set D = 'count_starts_vowel' / count Y = 'has_palindrome')  pos=132 ' =' => 
I'm counting lines where the first character is a vowel.

## Causality read-out — steer the pivot's goal encoding into an empty function

Same GoalScope steering, but instead of asking "what are you doing?", the goal-head encoding is steered into an
information-less function stub and the model continues the code. Mirrors `eval_goalscope --mode causality` (there:
the C++ stub `bool check(auto value){`). Here the destination is Python, so the stub reuses DST's variable name and
datatype: `def func({var}: {type}):` + `result =`. The continuation is the transferred *idea* — for the patched
run, does it implement SRC's target instead of DST's frontier?

In [23]:
from src.goals.goalscope import apply_goalscope, GOALSCOPE_SYS

CAUSAL_MAX_NEW = 64
CAUSAL_INSTRUCTION_PY = ("You are implementing a simple Python function. Just write the function body, without any "
                         "comments or explanations.")
# Python stub from DST's own signature (same variable name + datatype as the destination prompt)
CAUSAL_PREFILL = DATA._signature(DATATYPE, dst_inst.var_name, "python") + "\n    return"


def _chat(user, assistant):
    msgs = [{"role": "system", "content": GOALSCOPE_SYS}, {"role": "user", "content": user},
            {"role": "assistant", "content": assistant}]
    kw = dict(add_generation_prompt=False, continue_final_message=True, tokenize=False)
    try:
        return mt.tokenizer.apply_chat_template(msgs, chat_template_kwargs={"enable_thinking": False}, **kw)
    except TypeError:
        return mt.tokenizer.apply_chat_template(msgs, **kw)


CAUSAL_GS_PROMPT = _chat(CAUSAL_INSTRUCTION_PY, CAUSAL_PREFILL)


def causal_readout(tag, prompt, pivot, iface=None):
    res = apply_goalscope(prompt=prompt, mt=mt, selected_heads=list(READOUT_HEADS), strategy=RO_KW["strategy"],
                          alpha=RO_KW["alpha"], steer_layer_range=RO_KW["layers"], batch_size=1,
                          max_new_tokens=CAUSAL_MAX_NEW, goalscope_prompt=CAUSAL_GS_PROMPT,
                          token_positions=[pivot], include_noncode=False, show=False, patch_interface=iface)
    out = res[0]["patched_gen"]
    rich_print(f"[bold]{tag}[/bold]  pos={res[0]['pos']} {res[0]['token']!r} =>\n"
               f"[grey58]{escape(CAUSAL_PREFILL)}[/grey58][italic]{escape(out)}[/italic]")
    return out


rich_print(f"[grey58]causal stub:[/grey58] {escape(CAUSAL_PREFILL)!s}")
CAUSAL = {
    "source":      causal_readout(f"source (target {TN(TARGET)})", src_prompt, piv_src),
    "destination": causal_readout(f"destination (frontier {TN(FRONTIER)})", dst_prompt, piv_dst_clean),
    "patched":     causal_readout(f"patched{' (kv + q)' if USE_KV else ''} (expect {EXPECT_PATCHED})", dst_tok, piv_dst,
                                  iface=IFACE),
}
if USE_KV:
    CAUSAL["kv"] = causal_readout(f"kv-only (expect {TN(KV_NEXT)})", dst_tok, piv_dst, iface=ro_iface())

causal stub: def func(words: list[str]):
    return

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

source (target C = 'distinct_count')  pos=146 ' =' =>
def func(words: list[str]):
    return len(set(words))

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

destination (frontier X = 'total_length')  pos=129 ' =' =>
def func(words: list[str]):
    return sum(len(word) for word in words)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

patched (kv + q) (expect set D = 'count_starts_vowel' / count Y = 'has_palindrome')  pos=132 ' =' =>
def func(words: list[str]):
    return sum(1 for word in words if word[0].isupper())
def count_vowels(words: list[str]):
    return sum(1 for word in words if sum(1 for char in word if char.lower() in 'aeiou') > 0)
def count_long_words(words: list[str]):

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

kv-only (expect B = 'count_long')  pos=132 ' =' =>
def func(words: list[str]):
    return sum(1 for word in words if len(word) > 5)

## Gaze — where does the pivot look in the task list?

Mean value-weighted attention from the pivot onto each listed task's description, averaged over `GAZE_HEADS`.
*source* is read over SRC's list; *destination* and *patched* over DST's list. The patched gaze uses the
pivot-only patch, so it shows what the injected query attends to. SRC's target is absent from DST's list, so the
patched gaze can only show *which DST task* the transplanted query lands on.

In [24]:
def pivot_gaze(tok, items, piv, iface=None):
    # mean value-weighted pivot attention (GAZE_HEADS) per item; items = {letter: positions in `tok`}
    kw = {} if iface is None else dict(patch_interface=iface)
    info = get_attention_matrices(tok, mt, value_weighted=True, **kw)
    info = info[0] if isinstance(info, list) else info
    span = list(range(min(p[0] for p in items.values()), max(p[-1] for p in items.values()) + 1))
    a = pivot_attn(info, list(GAZE_HEADS), span, piv)                  # (len(span),)
    at = {p: v for p, v in zip(span, a)}
    per_item = {L: float(np.mean([at[p] for p in ps])) for L, ps in items.items()}
    return per_item, span, a

# gaze is measured under the plain patch (no OV boost — it scales the write, not the attention), as in 04.3
GAZE = {
    "source":      pivot_gaze(src_tok, item_positions(src_prompt, SRC_ORDER), piv_src),
    "destination": pivot_gaze(dst_tok_clean, item_positions(dst_prompt, DST_ORDER), piv_dst_clean),
}
if USE_KV:
    GAZE["kv"] = pivot_gaze(dst_tok, PATCHED_ITEMS, piv_dst, iface=ro_iface(boost=False))
GAZE["patched"] = pivot_gaze(dst_tok, PATCHED_ITEMS, piv_dst, iface=ro_iface(PF, boost=False))

t = Table(title=f"pivot gaze per task (mean value-weighted attn, {len(GAZE_HEADS)} heads) — argmax in bold"
                + (" · kv / patched read over p_kv's task span" if USE_KV else ""), highlight=False)
t.add_column("run"); t.add_column("expect")
for k in range(max(len(SRC_ORDER), len(DST_ORDER), len(KV_ORDER) if USE_KV else 0)):
    t.add_column(f"task {k + 1}", justify="right")
rows = [("source", TN(TARGET)), ("destination", TN(FRONTIER))]
rows += [("kv", TN(KV_NEXT)), ("patched", f"set {TN(H1_NEXT)} / count {TN(COUNT_NEXT)}")] if USE_KV else [("patched", f"{TN(TARGET)} (absent)")]
for run, expect in rows:
    g_ = GAZE[run][0]; top = max(g_, key=g_.get)
    t.add_row(run, expect, *[f"[bold]{L}={v:.4f}[/bold]" if L == top else f"{L}={v:.4f}" for L, v in g_.items()])
rich_print(t)

# token-level views: clean DST over its own list; patched over DST's (kv-patched) task span
_, span_c, a_clean = GAZE["destination"]
toks_c = [mt.tokenizer.decode([int(dst_tok_clean["input_ids"][0][j])]) for j in span_c]
show_delta_tokens(toks_c, a_clean, "destination · pivot → DST tasks (clean)")
_, span, a_patched = GAZE["patched"]
toks = [patched_tok_text(j) for j in span]            # p_kv's task text when USE_KV
if USE_KV:
    show_delta_tokens(toks, GAZE["kv"][2], "kv-only · pivot → p_kv tasks")
show_delta_tokens(toks, a_patched, f"patched · pivot → {PATCHED_LABEL}")
if not USE_KV:
    show_delta_tokens(toks, a_patched - a_clean, "patched − clean · pivot → DST tasks")

 pivot gaze per task (mean value-weighted attn, 250 heads) — argmax in bold · kv / patched read over p_kv's 
                                                 task span                                                  
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┓
┃ run         ┃ expect                                                    ┃   task 1 ┃   task 2 ┃   task 3 ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━┩
│ source      │ C = 'distinct_count'                                      │ A=0.0328 │ C=0.1814 │ B=0.0526 │
│ destination │ X = 'total_length'                                        │ A=0.0462 │ X=0.1879 │ Y=0.0606 │
│ kv          │ B = 'count_long'                                          │ A=0.0413 │ B=0.1401 │ D=0.0589 │
│ patched     │ set D = 'count_starts_vowel' / count Y = 'has_palindrome' │ A=0.0441 │ B=0.0520 │ D=0.0989 │
└─────────────┴───────────────────────────────────────────────────────────┴──────────┴──────────┴──────────┘

destination · pivot → DST tasks (clean)   (green +Δ / red −Δ, |Δ|max=0.573)

kv-only · pivot → p_kv tasks   (green +Δ / red −Δ, |Δ|max=0.353)

patched · pivot → p_kv tasks   (green +Δ / red −Δ, |Δ|max=0.304)

## Closed-loop query transfer — regenerate the goal heads' query at every step

Instead of re-clamping the same SRC query at every decode step:

1. **Prefill:** patch the SRC queries at `PREFILL_ANCHORS` into DST (as above) → first token `t1`.
2. **Goal encoding:** the goal heads' OV write in that q-patched DST prefill (the pass that emits `t1`), at ` =`
   (`GOAL_ENC="eq"`) or averaged over `PREFILL_ANCHORS` (`"mean"`).
3. **Each step k ≥ 1:** append `t_k` to the *probe* — an information-less prompt ending
   `def func({var}: {type}):` / `    return` (`PIVOT="var_eq"`) or the bare trailing indent (`PIVOT="indent"`) — with the goal encoding steered in (`additive_norm`, α 0.1, residual
   after layers 35–45) at the probe pivot (` return` / the indent) and, if `PROBE_STEER="all"`, at every appended token too. Read the probe's query
   at `t_k` (`q_caus`, goal heads only), write it into the patched run's query at `t_k`, and generate `t_{k+1}`.

The goal stays fixed; the query is recomputed from (goal + tokens so far). Stops at the first newline or
`LOOP_MAX` tokens.

In [25]:
# ── knobs ────────────────────────────────────────────────────────────────────
GOAL_ENC    = "mean"         # "eq": goal heads' OV write at DST " =" · "mean": mean over PREFILL_ANCHORS
PROBE_STEER = "all"        # "pivot": steer only the probe's " return" · "all": also every appended token
LOOP_MAX    = 25           # max generated tokens (also stops at the first newline)
PROBE_ALPHA, PROBE_LAYERS = 0.1, (35, 46)   # additive_norm into the residual after layers range(35, 46)

import baukit
from contextlib import contextmanager, nullcontext
from src.goals.goalscope import GOALSCOPE_SYS
from src.goals.head_loc_attn import collect_head_steering_vecs

# (2) goal encoding — goal heads' OV write in the q-patched DST prefill (IFACE = the PREFILL_ANCHORS patch)
_anch = list(PREFILL_ANCHORS)
_ov = collect_head_steering_vecs(mt=mt, src_tokenized=dst_tok, selected_heads=list(READOUT_HEADS),
                                 token_positions=[DST_ANCH[a] for a in _anch], patch_interface=IFACE)[0]
if GOAL_ENC == "eq":
    assert "eq" in _anch, "GOAL_ENC='eq' needs 'eq' in PREFILL_ANCHORS"
    _g = _ov[_anch.index("eq")]
elif GOAL_ENC == "mean":
    _g = _ov.mean(0)
else:
    raise ValueError(f"GOAL_ENC must be 'eq' or 'mean', got {GOAL_ENC!r}")
G_UNIT = (_g / _g.norm()).float()                     # additive_norm: unit direction, magnitude from ||h||

# probe: information-less function stub, same variable name + datatype as DST
# pivot shape follows PIVOT: "indent" -> bare trailing indent (else the model writes `return b = ...`),
# "var_eq" -> `return`
PROBE_PREFILL = DATA._signature(DATATYPE, dst_inst.var_name, "python") + ("\n    return" if PIVOT == "var_eq" else "\n    ")
PROBE_INSTRUCTION = ("You are implementing a simple Python function. Just write the function body, without any "
                     "comments or explanations.")


def _probe_chat(user, assistant):
    msgs = [{"role": "system", "content": GOALSCOPE_SYS}, {"role": "user", "content": user},
            {"role": "assistant", "content": assistant}]
    kw = dict(add_generation_prompt=False, continue_final_message=True, tokenize=False)
    try:
        return mt.tokenizer.apply_chat_template(msgs, chat_template_kwargs={"enable_thinking": False}, **kw)
    except TypeError:
        return mt.tokenizer.apply_chat_template(msgs, **kw)


probe_prompt = _probe_chat(PROBE_INSTRUCTION, PROBE_PREFILL)
probe_tok = prepare_input(prompts=probe_prompt, tokenizer=mt)
piv_probe = probe_tok["input_ids"].shape[1] - 1
print(f"probe pivot {mt.tokenizer.decode(probe_tok['input_ids'][0][piv_probe:piv_probe + 1])!r} · "
      f"goal enc = {GOAL_ENC} over {_anch} · probe steer = {PROBE_STEER} · layers {PROBE_LAYERS}")

_STEER_NAMES = [mt.layer_name_format.format(l) for l in range(*PROBE_LAYERS)]
_Q_NAMES = {l: FUNC._proj_name(l, "q", mt) for l in Q_LAYERS}


@contextmanager
def goal_steer(positions):
    # h <- h + alpha * ||h|| * g_unit at `positions` of the current forward, residual after each steer layer
    def edit(output, layer):
        h = output[0] if isinstance(output, tuple) else output
        for p in positions:
            v = h[:, p, :]
            h[:, p, :] = v + PROBE_ALPHA * v.float().norm(dim=-1, keepdim=True).to(v.dtype) \
                * G_UNIT.to(v.device, v.dtype)
        return output
    with baukit.TraceDict(mt._model, _STEER_NAMES, edit_output=edit):
        yield


def _ext(am):
    return torch.cat([am, torch.ones_like(am[:, :1])], 1)


def _head_cos(qa, qb):
    # mean over PATCH_HEADS of cos(q_a, q_b) per head slice
    hd, out = mt.head_dim, []
    for l, h in PATCH_HEADS:
        a, b = qa[l][0, h * hd:(h + 1) * hd], qb[l][0, h * hd:(h + 1) * hd]
        out.append(float(torch.nn.functional.cosine_similarity(a, b, dim=0)))
    return float(np.mean(out))


@torch.no_grad()
def closed_loop(verbose=True):
    eos = mt.tokenizer.eos_token_id
    # (1) patched-run prefill: SRC queries at the anchors -> t1
    ids, am = dst_tok["input_ids"], dst_tok["attention_mask"]
    with patched_ctx(pf=PF):
        out = mt._model(input_ids=ids, attention_mask=am, use_cache=True)
    pkv_d, nxt = out.past_key_values, out.logits[:, -1].argmax(-1, keepdim=True)
    # probe prefill: goal steered at " return"
    p_am = probe_tok["attention_mask"]
    with goal_steer([piv_probe]):
        pout = mt._model(input_ids=probe_tok["input_ids"], attention_mask=p_am, use_cache=True)
    pkv_p = pout.past_key_values

    gen, steps = [int(nxt)], []
    while len(gen) < LOOP_MAX and int(nxt) != eos and "\n" not in mt.tokenizer.decode(gen):
        # (3) probe consumes t_k (goal-steered if PROBE_STEER="all"); read q_caus at t_k on the goal heads
        p_am = _ext(p_am)
        ctx = goal_steer([-1]) if PROBE_STEER == "all" else nullcontext()
        with ctx, baukit.TraceDict(mt._model, list(_Q_NAMES.values()), retain_output=True) as tr:
            pout = mt._model(input_ids=nxt, attention_mask=p_am, past_key_values=pkv_p, use_cache=True)
        pkv_p = pout.past_key_values
        q_caus = {l: tr[n].output[0, -1:].detach().float().cpu() for l, n in _Q_NAMES.items()}
        # patched run: q_caus replaces the goal heads' query at t_k -> t_{k+1}
        am = _ext(am)
        with patched_ctx(pf=[dict(which="q", cache=q_caus, heads=_qh, positions=[-1])]):
            out = mt._model(input_ids=nxt, attention_mask=am, past_key_values=pkv_d, use_cache=True)
        pkv_d = out.past_key_values
        steps.append(dict(k=len(gen), tok=mt.tokenizer.decode(gen[-1:]), cos_q_dec=_head_cos(q_caus, q_dec), q=q_caus))
        nxt = out.logits[:, -1].argmax(-1, keepdim=True)
        gen.append(int(nxt))
    text = mt.tokenizer.decode(gen, skip_special_tokens=True)
    if verbose:
        t = Table(title=f"closed loop · per step: token t_k whose query is replaced · cos(q_caus, decode q of the "
                        f"re-clamp run), mean over {len(PATCH_HEADS)} heads", highlight=False)
        t.add_column("k", justify="right"); t.add_column("t_k"); t.add_column("cos", justify="right")
        for s in steps:
            t.add_row(str(s["k"]), repr(s["tok"]), f"{s['cos_q_dec']:.3f}")
        rich_print(t)
    return text, steps, gen


GEN["closed_loop"], LOOP_STEPS, LOOP_IDS = closed_loop()
_line = lambda s: s.split("\n")[0]
rich_print(f"[sea_green3]destination (frontier {TN(FRONTIER)}):[/sea_green3] {escape(_line(GEN['destination']))}")
rich_print(f"[salmon1]q_src re-clamp (N_STEER={N_STEER}):[/salmon1]   {escape(_line(GEN['patched']))}")
rich_print(f"[bold magenta]closed loop (expect {EXPECT_PATCHED}):[/bold magenta]  {escape(_line(GEN['closed_loop']))}")

probe pivot ' return' · goal enc = mean over ['indent', 'var', 'eq'] · probe steer = all · layers (35, 46)


 closed loop · per step: 
token t_k whose query is 
 replaced · cos(q_caus,  
decode q of the re-clamp 
run), mean over 250 heads
┏━━━━┳━━━━━━━━━━┳━━━━━━━┓
┃  k ┃ t_k      ┃   cos ┃
┡━━━━╇━━━━━━━━━━╇━━━━━━━┩
│  1 │ ' sum'   │ 0.662 │
│  2 │ '('      │ 0.643 │
│  3 │ '1'      │ 0.653 │
│  4 │ ' for'   │ 0.640 │
│  5 │ ' w'     │ 0.609 │
│  6 │ ' in'    │ 0.676 │
│  7 │ ' words' │ 0.627 │
│  8 │ ' if'    │ 0.643 │
│  9 │ ' w'     │ 0.611 │
│ 10 │ '['      │ 0.582 │
│ 11 │ '0'      │ 0.590 │
│ 12 │ '].'     │ 0.606 │
│ 13 │ 'lower'  │ 0.593 │
│ 14 │ '()'     │ 0.598 │
│ 15 │ ' in'    │ 0.637 │
│ 16 │ " '"     │ 0.603 │
│ 17 │ 'ae'     │ 0.584 │
│ 18 │ 'iou'    │ 0.604 │
└────┴──────────┴───────┘

destination (frontier X = 'total_length'):  sum(len(w) for w in words)

q_src re-clamp (N_STEER=32):    sum(1 for w in words if len(w) > 0)

closed loop (expect set D = 'count_starts_vowel' / count Y = 'has_palindrome'):   sum(1 for w in words if 
w[0].lower() in 'aeiou')

### Closed loop — read-outs of the goal encoding (step ii) + gaze along the loop

* **GoalScope** ("what are you doing?") and **causality** (the `return` probe, free-running) read-outs of the goal
  encoding `G_UNIT` itself — what the vector that drives `q_caus` says on its own. The causality read-out steers the
  probe the same way the loop does (`PROBE_STEER`).
* **Gaze** of the patched run over DST's task list, at the pivot (the step-(i) prefill patch) and at every loop token
  `t_k` (its goal-head query = `q_caus`). Obtained by replaying DST + `t_1..t_m` with the same patches (prefill anchors
  + `q_caus` at each `t_k`), which reproduces the loop's forward exactly.

In [26]:
from src.goals.goalscope import head_additive_generate, _build_goalscope_prompt
from src.utils.typing import TokenizerOutput

# ── read-outs of the goal encoding ───────────────────────────────────────────
_ro = lambda prompt, n_steer: head_additive_generate(
    mt=mt, dest_inputs=prepare_input(prompts=prompt, tokenizer=mt), steering_vecs=_g.float(),
    layers=list(range(*PROBE_LAYERS)), max_new_tokens=64, strategy="additive_norm", n_steer_tokens=n_steer,
    alpha=PROBE_ALPHA)
LOOP_RO = {
    "goalscope": _ro(_build_goalscope_prompt(mt), None),
    "causality": _ro(probe_prompt, None if PROBE_STEER == "all" else 1),
}
rich_print(f"[bold]goal encoding ({GOAL_ENC}) · GoalScope[/bold] => [italic]{escape(LOOP_RO['goalscope'])}[/italic]")
rich_print(f"[bold]goal encoding ({GOAL_ENC}) · causality[/bold] =>\n[grey58]{escape(PROBE_PREFILL)}[/grey58]"
           f"[italic]{escape(LOOP_RO['causality'])}[/italic]")

# ── gaze along the loop ─────────────────────────────────────────────────────
L0 = dst_tok["input_ids"].shape[1]
m = len(LOOP_STEPS)        # steps whose query was replaced: t_1 .. t_m
replay = TokenizerOutput(data=dict(
    input_ids=torch.cat([dst_tok["input_ids"], torch.tensor([LOOP_IDS[:m]], device=dst_tok["input_ids"].device)], 1),
    attention_mask=torch.ones(1, L0 + m, dtype=dst_tok["attention_mask"].dtype,
                              device=dst_tok["attention_mask"].device)))
loop_spec = dict(which="q", heads=_qh, positions=[L0 + k for k in range(m)],
                 cache={l: torch.cat([s["q"][l] for s in LOOP_STEPS], 0) for l in Q_LAYERS})
_info = get_attention_matrices(replay, mt, value_weighted=True,
                               patch_interface=ro_iface(PF + [loop_spec], boost=False))
_info = _info[0] if isinstance(_info, list) else _info

_items = PATCHED_ITEMS                    # DST items (or KV items when USE_KV)
_span = list(range(min(p[0] for p in _items.values()), max(p[-1] for p in _items.values()) + 1))
def _per_item(a):
    at = dict(zip(_span, a)); return {L: float(np.mean([at[p] for p in ps])) for L, ps in _items.items()}

LOOP_GAZE = [("pivot " + repr(mt.tokenizer.decode(dst_tok["input_ids"][0][piv_dst:piv_dst + 1])),
              pivot_attn(_info, list(GAZE_HEADS), _span, piv_dst))]
LOOP_GAZE += [(f"t_{s['k']} {s['tok']!r}", pivot_attn(_info, list(GAZE_HEADS), _span, L0 + s["k"] - 1))
              for s in LOOP_STEPS]
t = Table(title=f"closed-loop gaze over {PATCHED_LABEL} (mean value-weighted attn, {len(GAZE_HEADS)} heads; frontier "
                f"{TN(FRONTIER)}, SRC target {TN(TARGET)}" + (f", kv-only {TN(KV_NEXT)}, set {TN(H1_NEXT)} / count {TN(COUNT_NEXT)}" if USE_KV else " absent") + ") — argmax in bold", highlight=False)
t.add_column("query"); [t.add_column(L, justify="right") for L in _items]
for name, a in LOOP_GAZE:
    g_ = _per_item(a); top = max(g_, key=g_.get)
    t.add_row(name, *[f"[bold]{v:.4f}[/bold]" if L == top else f"{v:.4f}" for L, v in g_.items()])
rich_print(t)

_toks = [patched_tok_text(j) for j in _span]           # p_kv's task text when USE_KV
# actual gaze (not delta) of the goal heads at every loop step t_k, whose query was replaced by q_caus
for name, a in LOOP_GAZE[1:]:
    show_delta_tokens(_toks, a, f"closed loop · {name} (query = q_caus) → {PATCHED_LABEL}")

goal encoding (mean) · GoalScope => I'm counting lines where the first character is uppercase.

goal encoding (mean) · causality =>
def func(words: list[str]):
    return sum(1 for word in words if word[0].isupper())
    count = 0
    for word in words:
        if word.endswith('e'):
            count += 1
    return count
    return len([w for w in words if 'a' in w])

  closed-loop gaze over p_kv tasks (mean   
value-weighted attn, 250 heads; frontier X 
     = 'total_length', SRC target C =      
'distinct_count', kv-only B = 'count_long',
 set D = 'count_starts_vowel' / count Y =  
    'has_palindrome') — argmax in bold     
┏━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┓
┃ query        ┃      A ┃      B ┃      D ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━┩
│ pivot ' ='   │ 0.0440 │ 0.0521 │ 0.0990 │
│ t_1 ' sum'   │ 0.1456 │ 0.1049 │ 0.0935 │
│ t_2 '('      │ 0.1532 │ 0.1193 │ 0.1046 │
│ t_3 '1'      │ 0.0943 │ 0.0707 │ 0.0652 │
│ t_4 ' for'   │ 0.1440 │ 0.0831 │ 0.0948 │
│ t_5 ' w'     │ 0.1209 │ 0.0884 │ 0.0935 │
│ t_6 ' in'    │ 0.0562 │ 0.0658 │ 0.0519 │
│ t_7 ' words' │ 0.0941 │ 0.0907 │ 0.0681 │
│ t_8 ' if'    │ 0.2192 │ 0.0977 │ 0.1520 │
│ t_9 ' w'     │ 0.2836 │ 0.1124 │ 0.1569 │
│ t_10 '['     │ 0.1971 │ 0.0845 │ 0.1343 │
│ t_11 '0'     │ 0.2117 │ 0.0637 │ 0.1289 │
│ t_12 '].'    │ 0.2291 │ 0.0543 │ 0.1605 │
│ t_13 'lower' │ 0.1146 │ 0.0470 │ 0.1289 │
│ t_14 '()'    │ 0.1239 │ 0.0737 │ 0.1142 │
│ t_15 ' in'   │ 0.0908 │ 0.0401 │ 0.1559 │
│ t_16 " '"    │ 0.1269 │ 0.0528 │ 0.1651 │
│ t_17 'ae'    │ 0.0864 │ 0.0502 │ 0.0810 │
│ t_18 'iou'   │ 0.0761 │ 0.0562 │ 0.0548 │
└──────────────┴────────┴────────┴────────┘

closed loop · t_1 ' sum' (query = q_caus) → p_kv tasks   (green +Δ / red −Δ, |Δ|max=0.416)

closed loop · t_2 '(' (query = q_caus) → p_kv tasks   (green +Δ / red −Δ, |Δ|max=0.525)

closed loop · t_3 '1' (query = q_caus) → p_kv tasks   (green +Δ / red −Δ, |Δ|max=0.239)

... (11 outputs left out)

closed loop · t_15 ' in' (query = q_caus) → p_kv tasks   (green +Δ / red −Δ, |Δ|max=1.183)

closed loop · t_16 " '" (query = q_caus) → p_kv tasks   (green +Δ / red −Δ, |Δ|max=1.260)

closed loop · t_17 'ae' (query = q_caus) → p_kv tasks   (green +Δ / red −Δ, |Δ|max=0.515)

closed loop · t_18 'iou' (query = q_caus) → p_kv tasks   (green +Δ / red −Δ, |Δ|max=0.269)